# Test the deployed tire calculator

## Goal

Open the published calculator in Chromium and verify that the Python model
loads, renders every valid tire pairing and recalculates after an input change.

In [1]:
import json
import os
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread

from playwright.async_api import async_playwright

## Setup

The deployment workflow supplies the exact GitHub Pages URL and commit query.

In [2]:
calculator_url = os.environ["CALCULATOR_URL"]

if calculator_url == "local":
    local_site_server = ThreadingHTTPServer(
        ("127.0.0.1", 8765),
        partial(SimpleHTTPRequestHandler, directory="../docs"),
    )
    local_site_thread = Thread(
        target=local_site_server.serve_forever,
        daemon=True,
    )
    local_site_thread.start()
    calculator_url = "http://127.0.0.1:8765/"

## Test

The browser must reach the ready state without page errors. The test then
changes power from 200 W to 250 W and requires the predicted speed to
change.

In [3]:
browser_errors = []

async with async_playwright() as playwright:
    browser = await playwright.chromium.launch()
    page = await browser.new_page()
    page.on(
        "pageerror",
        lambda error: browser_errors.append(str(error)),
    )

    response = await page.goto(
        calculator_url,
        wait_until="domcontentloaded",
        timeout=120_000,
    )
    assert response is not None and response.ok
    await page.locator('body[data-ready="true"]').wait_for(
        timeout=120_000
    )
    real_hpms_response = await page.evaluate(
        """async () => {
            const parameters = new URLSearchParams({
                "$select": "iri,line",
                "$where": "iri IS NOT NULL AND within_circle(line, 44.2989, -74.0989, 250)",
                "$limit": "5"
            });
            const response = await fetch(
                "https://data.transportation.gov/resource/42um-tgh5.json?" + parameters
            );
            return {
                status: response.status,
                rows: await response.json()
            };
        }"""
    )
    assert real_hpms_response["status"] == 200
    assert any("iri" in row for row in real_hpms_response["rows"])
    real_weather_response = await page.evaluate(
        """async () => {
            const parameters = new URLSearchParams({
                latitude: "44.2989",
                longitude: "-74.0989",
                start_date: "2025-07-01",
                end_date: "2025-07-01",
                hourly: "temperature_2m,surface_pressure,wind_speed_10m,wind_direction_10m",
                timezone: "auto"
            });
            const response = await fetch(
                "https://archive-api.open-meteo.com/v1/archive?" + parameters
            );
            return {
                status: response.status,
                weather: await response.json()
            };
        }"""
    )
    assert real_weather_response["status"] == 200
    assert "surface_pressure" in real_weather_response["weather"]["hourly"]
    assert "Power at pedals" in await page.locator(
        'label:has(#rider-power)'
    ).inner_text()
    assert await page.locator("#rider-power").input_value() == "200"
    assert await page.locator("#wheel-size").input_value() == "622"
    assert await page.locator("#speed-from").count() == 0
    assert await page.locator("#speed-to").count() == 0
    assert await page.get_by_text("Wind exposure").count() == 0
    model_note = await page.locator(".model-note").inner_text()
    assert "not a 105% rule" in model_note
    assert "29.8 and 31.4 mm" in model_note
    assert "aero baseline per tire" in model_note

    sources_panel = page.locator('[data-testid="data-sources"]')
    assert await sources_panel.count() == 1
    sources_text = await sources_panel.inner_text()
    assert "Bicycle Rolling Resistance" in sources_text
    assert "Parcours’ 2026 wind-tunnel test" in sources_text
    assert "not yet yaw-resolved" in sources_text
    assert await sources_panel.locator("a").count() >= 6

    winner_card = page.locator('[data-testid="winner-card"]')
    await winner_card.wait_for()
    show_work_button = page.locator("#show-work-button")
    work_panel = page.locator('[data-testid="work-panel"]')
    assert await show_work_button.is_enabled()
    assert await work_panel.is_hidden()
    await show_work_button.click()
    assert await work_panel.is_visible()
    work_text = await work_panel.inner_text()
    assert "Power balance" in work_text
    assert "Crrref" in work_text
    assert "TT TR and S TR share" in work_text
    assert await show_work_button.inner_text() == "Hide your work"

    ranking_rows = page.locator("#ranking-table tbody tr")
    ranking_row_count = await ranking_rows.count()
    assert ranking_row_count == 99
    assert await page.locator(".table-wrap").evaluate(
        "element => element.scrollHeight > element.clientHeight"
    )

    predicted_speed_before = await page.locator(
        '[data-testid="predicted-speed"]'
    ).text_content()
    await page.locator("#rider-power").fill("250")
    await page.locator("#calculate-button").click()
    await page.wait_for_function(
        """previousSpeed => {
            const speed = document.querySelector(
                '[data-testid="predicted-speed"]'
            );
            return speed && speed.textContent !== previousSpeed;
        }""",
        arg=predicted_speed_before,
        timeout=30_000,
    )
    predicted_speed_after = await page.locator(
        '[data-testid="predicted-speed"]'
    ).text_content()

    assert predicted_speed_before != predicted_speed_after
    await page.locator("#wheel-size").select_option("584")
    await page.locator("#bike").select_option("Gravel race")
    await page.locator("#surface").select_option("Firm gravel")
    await page.locator("#calculate-button").click()
    await page.wait_for_function(
        """previousSpeed => {
            const speed = document.querySelector(
                '[data-testid="predicted-speed"]'
            );
            return speed && speed.textContent !== previousSpeed;
        }""",
        arg=predicted_speed_after,
        timeout=30_000,
    )
    gravel_speed = await page.locator(
        '[data-testid="predicted-speed"]'
    ).text_content()
    assert gravel_speed != predicted_speed_after

    await page.route(
        "https://data.transportation.gov/resource/42um-tgh5.json**",
        lambda route: route.fulfill(
            content_type="application/json",
            body=json.dumps([
                {
                    "iri": "100",
                    "iri_d": "2024-05-01T00:00:00.000",
                    "line": {
                        "type": "LineString",
                        "coordinates": [
                            [-74.0, 43.99],
                            [-74.0, 44.03],
                        ],
                    },
                },
            ]),
        ),
    )
    gpx_source = """<?xml version=\"1.0\"?>
<gpx version=\"1.1\" creator=\"test\"><trk><trkseg>
<trkpt lat=\"44.000\" lon=\"-74.000\" />
<trkpt lat=\"44.010\" lon=\"-74.000\" />
<trkpt lat=\"44.020\" lon=\"-74.000\" />
</trkseg></trk></gpx>"""
    await page.locator("#route-file").set_input_files({
        "name": "lake-placid-sample.gpx",
        "mimeType": "application/gpx+xml",
        "buffer": gpx_source.encode(),
    })
    await page.wait_for_function(
        """() => document.querySelector("#route-status").textContent.includes(
            "lake-placid-sample.gpx is ready"
        )""",
        timeout=30_000,
    )
    assert "2 roughness samples" in await page.locator(
        "#route-status"
    ).text_content()
    await page.locator("#analyze-route-button").click()
    await page.wait_for_function(
        """() => document.querySelector("#route-status").textContent.includes(
            "2 of 2 samples matched"
        )""",
        timeout=30_000,
    )
    route_speed = await page.locator(
        '[data-testid="predicted-speed"]'
    ).text_content()
    route_status = await page.locator("#route-status").text_content()
    assert route_speed != gravel_speed
    assert "100 in/mi modeled IRI" in route_status

    weather_hourly = {
        "time": ["2026-07-20T" + f"{hour:02}:00" for hour in range(24)],
        "temperature_2m": [20.0] * 24,
        "relative_humidity_2m": [50.0] * 24,
        "surface_pressure": [1000.0] * 24,
        "wind_speed_10m": [18.0] * 24,
        "wind_direction_10m": [270.0] * 24,
    }
    await page.route(
        "https://archive-api.open-meteo.com/v1/archive**",
        lambda route: route.fulfill(
            content_type="application/json",
            body=json.dumps([
                {"hourly": weather_hourly},
                {"hourly": weather_hourly},
            ]),
        ),
    )
    await page.locator("#race-date").fill("2027-07-20")
    assert await page.locator("#race-start-time").input_value() == "07:00"
    await page.locator("#analyze-weather-button").click()
    await page.wait_for_function(
        """() => document.querySelector("#weather-status").textContent.includes(
            "Historic weather from 2026-07-20, 2025-07-20, 2024-07-20"
        )""",
        timeout=30_000,
    )
    weather_speed = await page.locator(
        '[data-testid="predicted-speed"]'
    ).text_content()
    weather_status = await page.locator("#weather-status").text_content()
    assert weather_speed != route_speed
    assert "air density" in weather_status
    assert "apparent-wind yaw" in weather_status
    assert await page.locator(
        '#runtime-status[data-state="error"]'
    ).count() == 0
    assert browser_errors == []

    winner_text = await winner_card.inner_text()
    await browser.close()

if "local_site_server" in globals():
    local_site_server.shutdown()

{
    "url": calculator_url,
    "real_hpms_match_count": len(real_hpms_response["rows"]),
    "real_weather_hours": len(real_weather_response["weather"]["hourly"]["time"]),
    "winner": winner_text,
    "ranking_rows": ranking_row_count,
    "speed_before": predicted_speed_before,
    "speed_after": predicted_speed_after,
    "gravel_speed": gravel_speed,
    "route_speed": route_speed,
    "route_status": route_status,
    "weather_speed": weather_speed,
    "weather_status": weather_status,
    "browser_errors": browser_errors,
}

127.0.0.1 - - [31/Jul/2026 11:49:58] "GET / HTTP/1.1" 200 -


{'url': 'http://127.0.0.1:8765/',
 'real_hpms_match_count': 5,
 'real_weather_hours': 24,
 'winner': 'FASTEST MODELED SYSTEM\nFRONT\nSL-R 28\n\n72.9 psi · 29.5 mm mounted\n\nREAR\nTT TR 28\n\n78.8 psi · 29.2 mm mounted\n\n22.8 mph modeled steady speed at 250.0 W',
 'ranking_rows': 99,
 'speed_before': '23.8 mph',
 'speed_after': '25.8 mph',
 'gravel_speed': '22.3 mph',
 'route_speed': '22.5 mph',
 'route_status': '1.4 mi route · 2 of 2 samples matched · 100% HPMS coverage · 100 in/mi modeled IRI · measured 2024.',
 'weather_speed': '22.8 mph',
 'weather_status': 'Historic weather from 2026-07-20, 2025-07-20, 2024-07-20 at 2 course positions · 1.183 kg/m³ air density · 26.2° mean apparent-wind yaw at 22.8 mph.',
 'browser_errors': []}

## Result

A completed notebook is evidence that the deployed browser runtime, Python
model, result rendering and recalculation path all worked together.